# 03 Run the whole system

One notebook, top to bottom: shared feature store, local / pooled / federated baselines, the privacy dial, drift, forgetting and protection, and a membership-guessing stress test.

**Needs:** the repository folder (`src/fedfs`). In Colab, clone the repo or upload it and `%cd` into it first. Nothing else to install. It uses only NumPy, pandas, scikit-learn, SciPy and Matplotlib.

Takes roughly 20 to 40 minutes for 10 seeds and both models; use `--seeds 3` for a quick check.

## Step 1: Point Python at the code and get the data

In [ ]:
import os, sys
sys.path.insert(0, 'src')
from fedfs.data import download_uci, load_uci
download_uci('data/raw')
frames = load_uci('data/raw')
{h: len(d) for h, d in frames.items()}

## Step 2: The feature store

One agreed schema. The tables show how many values fell outside the valid range at each hospital (cholesterol 0 means 'not recorded') and how much is missing after cleaning.

In [ ]:
from fedfs import feature_store as fs
display(fs.validation_report(frames))
display(fs.missing_report(frames))
pd_schema = __import__('pandas').DataFrame([f.__dict__ for f in fs.SCHEMA])
display(pd_schema)

**Check before moving on.** If any count in the first table is surprising (for example chest_pain or thal being removed at one hospital), widen that range in `src/fedfs/feature_store.py` and re-run. The schema is the single source of truth.

## Step 3: Run everything

Local grid, pooled ceiling, federated with and without the shared feature store, epsilon sweep, drift and forgetting, membership test. Results land in `results/`.

In [ ]:
from fedfs.run_all import main
main(['--raw', 'data/raw', '--out', 'results', '--seeds', '10'])

## Step 4: Look at the figures

In [ ]:
from IPython.display import Image, display
for f in ['local_grid', 'privacy_curve', 'drift_forgetting']:
    display(Image(f'results/logistic/figures/{f}.png'))

## Step 5: Before you write anything down

Open `results/summary.md`. For every claim you want to make, mark it **proven** (it follows from the privacy accounting or the drift bound under stated assumptions, see `docs/theory.md`) or **measured** (we saw it in these experiments).

Then paste one row per experiment into the Results Log, failures included.